In [ ]:

from AQUA.batchAQUA_general import batchAQUA
from AQUA.AQUA_general import AQUA
from AQUA.stimulus import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

from simulation import get_F

In [2]:
IB = {'name': 'IB', 'C': 150, 'k': 1.2, 'v_r': -75, 'v_t': -45, 'v_peak': 50,
     'a': 0.01, 'b': 5, 'c': -56, 'd': 130, 'e': 0., 'f': 0., 'tau': 0.}

neuron = AQUA(IB)

print(neuron.get_params())



thresh, x_ini = neuron.get_threshold()

print(thresh)
print(x_ini)

{'name': 'IB', 'k': 1.2, 'C': 150, 'v_r': -75, 'v_t': -45, 'v_peak': 50, 'a': 0.01, 'b': 5, 'c': -56, 'd': 130, 'e': 0.0, 'f': 0.0, 'tau': 0.0}


100%|██████████| 39999/39999 [00:06<00:00, 6188.11it/s]

[308.46464646]
[-63.81466681  55.92666594   0.        ]


In [3]:
pdf = []
for i in range(10):
    pdf.append(IB)

df = pd.DataFrame(pdf)

print(df)

  name    C    k  v_r  v_t  v_peak     a  b   c    d    e    f  tau
0   IB  150  1.2  -75  -45      50  0.01  5 -56  130  0.0  0.0  0.0
1   IB  150  1.2  -75  -45      50  0.01  5 -56  130  0.0  0.0  0.0
2   IB  150  1.2  -75  -45      50  0.01  5 -56  130  0.0  0.0  0.0
3   IB  150  1.2  -75  -45      50  0.01  5 -56  130  0.0  0.0  0.0
4   IB  150  1.2  -75  -45      50  0.01  5 -56  130  0.0  0.0  0.0
5   IB  150  1.2  -75  -45      50  0.01  5 -56  130  0.0  0.0  0.0
6   IB  150  1.2  -75  -45      50  0.01  5 -56  130  0.0  0.0  0.0
7   IB  150  1.2  -75  -45      50  0.01  5 -56  130  0.0  0.0  0.0
8   IB  150  1.2  -75  -45      50  0.01  5 -56  130  0.0  0.0  0.0
9   IB  150  1.2  -75  -45      50  0.01  5 -56  130  0.0  0.0  0.0


In [4]:
spikes = np.array([[np.nan, np.nan],
                   [np.nan, np.nan],
                   [np.nan, np.nan],
                   [0.1, np.nan],
                   [0.1, 0.2],
                   [0.2, 0.3]])


print(spikes[0])
print(spikes[3][~np.isnan(spikes[3])])

def get_F(spikes, instant = False):
    """
    Returns an array of the desired firing frequency
    
    :param spikes: array of spike times
    :param instant: boolean, whether to get instantaneous firing frequency or not

    """
    N_neurons = len(spikes)
    F = np.zeros(N_neurons)

    for n in range(N_neurons):
        print(n)
        if np.isnan(spikes[n]).all() or np.sum(~np.isnan(spikes[n])) == 1:      # if no spikes or 1 spike
            F[n] = np.nan
        else:
            if instant:     # get instant firing frequency
                F[n] = 1000/(spikes[n][1] - spikes[n][0])           # first and second spikes
            else:           # get steady firing frequency (might be same as initial)
                spike_times = spikes[n][~np.isnan(spikes[n])]
                F[n] = 1000/(spike_times[-1] - spike_times[-2])     # last and second-to-last spikes
        
    
    return F

    
print(get_F(spikes))

[nan nan]
[0.1]
0
1
2
3
4
5
[   nan    nan    nan    nan 10000. 10000.]


In [30]:
filename = "RS_intHD_test.pickle"
with open(filename, 'rb') as file:
    df = pickle.load(file)

In [31]:
print(df.head())
print("- - - - ")
print(df.groupby(['f', 'e', 'tau'])['I_h'].mean())

     e     f  tau  I_h  F_instant  F_steady  autapse current  autapse delay
0  0.0   0.0  0.0  0.0        NaN       NaN              NaN            NaN
1  0.1  50.0  0.0  0.0        NaN       NaN            500.0       6.931472
2  0.1  50.0  1.0  0.0        NaN       NaN            500.0       7.931472
3  0.1  50.0  2.0  0.0        NaN       NaN            500.0       8.931472
4  0.1  50.0  3.0  0.0        NaN       NaN            500.0       9.931472
- - - - 
f      e    tau
0.0    0.0  0.0    250.0
50.0   0.1  0.0    250.0
            1.0    250.0
            2.0    250.0
            3.0    250.0
                   ...  
250.0  0.5  0.0    250.0
            1.0    250.0
            2.0    250.0
            3.0    250.0
            4.0    250.0
Name: I_h, Length: 501, dtype: float64


In [35]:
df['temp_id'] = df.groupby(['e', 'f', 'tau']).ngroup()

print(df.groupby(['e', 'f', 'tau'])['temp_id'].mean())

e    f      tau
0.0  0.0    0.0      0.0
0.1  50.0   0.0      1.0
            1.0      2.0
            2.0      3.0
            3.0      4.0
                   ...  
0.5  250.0  0.0    496.0
            1.0    497.0
            2.0    498.0
            3.0    499.0
            4.0    500.0
Name: temp_id, Length: 501, dtype: float64
